##### Copyright 2025 Perceptron AI.

In [ ]:
# Licensed under the MIT License (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://opensource.org/licenses/MIT
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Capability — Video Clipping
Find the exact moment an event occurs in a video and return a temporal clip (start/end timestamps) grounding the answer.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/isaac-0.3-max/video-clipping.ipynb)

<video src="https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/video-clipping/mj_shot_short.mp4" controls width="640" muted loop>
  Your browser doesn't support the video tag.
</video>

## Install dependencies

In [ ]:
%pip install --upgrade perceptron --quiet

## Download the sample video

In [ ]:
VIDEO_URL = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/video-clipping/mj_shot_short.mp4"

!curl -L -so mj_shot_short.mp4 {VIDEO_URL}

## Configure the Perceptron client

In [ ]:
import os

from perceptron import configure, question, video

api_key = os.getenv("PERCEPTRON_API_KEY", "<your Perceptron API key>")
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="isaac-0.3-max",
    api_key=api_key,
)

VIDEO_PATH = "mj_shot_short.mp4"

## Ask the model to clip the moment
Reasoning is on so the model can search through frames before localizing. `expects="clip"` parses any `<clip>` tags the model emits into structured `Clip` objects with start/end timestamps.

In [ ]:
QUESTION = "Clip the exact moment the ball passes through the hoop."

result = question(video(VIDEO_PATH), QUESTION, reasoning=True, expects="clip")

print("--- Reasoning ---")
print(result.reasoning or "(none)")
print("\n--- Answer ---")
print(result.text)

## Inspect the returned clips
Each `Clip` carries a `timestamp.at` (start, in seconds) and an optional `timestamp.until` (end). When `until` is `None`, the clip refers to a single moment rather than a range.

In [ ]:
clips = result.clips or []
if not clips:
    print("No clips returned. Try rephrasing the question or pointing at a different moment.")
else:
    print(f"Returned {len(clips)} clip(s):")
    for idx, clip in enumerate(clips, start=1):
        ts = clip.timestamp
        if ts.until is None:
            window = f"moment at {ts.at:.2f}s"
        else:
            window = f"{ts.at:.2f}s - {ts.until:.2f}s"
        label = clip.mention or "(no mention)"
        print(f"  Clip {idx}: {window} - {label}")

## Conclusion & next steps
- Try other event-localization prompts ("clip every save attempt," "clip when the package is dropped," "clip every shot on goal").
- Use `result.clips[i].timestamp.at` / `.until` to drive a video editor cut, generate a highlight reel, or label training data automatically.
- Pair with the [Video Q&A](https://github.com/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/isaac-0.3-max/video-qa.ipynb) notebook for an unstructured-text alternative when you don't need timestamps.